In [5]:
import json
import numpy as np
from pathlib import Path

BASE_DIR = Path("human_scores")

AUDITOR_DIMS = [
    "issue_identification",
    "reasoning_quality",
    "recommendation_actionability",
    "rubric_calibration",
    "overall_audit_quality"
]

ARCHITECT_DIMS = [
    "contract_alignment",
    "fix_report_accuracy",
    "architecture_quality",
    "security_coverage",
    "plan_completeness"
]

def load_jsonl(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def compute_averages(records, dims):
    scores = {d: [] for d in dims}
    for r in records:
        for d in dims:
            if d in r.get("scores", {}) and isinstance(r["scores"][d], (int, float)):
                scores[d].append(float(r["scores"][d]))
    return {d: (scores[d], float(np.mean(scores[d]))) if scores[d] else ([], None) for d in dims}

def process_user(user_path):
    results = {}

    auditor_file = user_path / "auditor_scores.jsonl"
    architect_file = user_path / "architect_scores.jsonl"

    if auditor_file.exists():
        results["auditor"] = compute_averages(load_jsonl(auditor_file), AUDITOR_DIMS)

    if architect_file.exists():
        results["architect"] = compute_averages(load_jsonl(architect_file), ARCHITECT_DIMS)

    return results


all_users_results = {}

for user_folder in BASE_DIR.iterdir():
    if user_folder.is_dir():
        user_name = user_folder.name
        user_results = process_user(user_folder)
        if user_results:
            all_users_results[user_name] = user_results


print("\n" + "="*60)
print("HUMAN EVALUATION RESULTS — PER AUTHOR")
print("="*60)

global_scores = {"auditor": {d: [] for d in AUDITOR_DIMS}, "architect": {d: [] for d in ARCHITECT_DIMS}}

output = {}

for user, res in all_users_results.items():
    print(f"\nUser: {user}")
    output[user] = {}

    if "auditor" in res:
        print("\n  AUDITOR:")
        output[user]["auditor"] = {}
        for dim, (raw, val) in res["auditor"].items():
            if val is not None:
                bar = "#" * int(val * 3)
                print(f"    {dim:35s}: {val:.2f}/10  {bar}")
                output[user]["auditor"][dim] = round(val, 4)
                global_scores["auditor"][dim].extend(raw)

    if "architect" in res:
        print("\n  ARCHITECT:")
        output[user]["architect"] = {}
        for dim, (raw, val) in res["architect"].items():
            if val is not None:
                bar = "#" * int(val * 3)
                print(f"    {dim:35s}: {val:.2f}/10  {bar}")
                output[user]["architect"][dim] = round(val, 4)
                global_scores["architect"][dim].extend(raw)


print("\n" + "="*60)
print("OVERALL AVERAGE (ALL AUTHORS COMBINED)")
print("="*60)

overall = {}

print("\n  AUDITOR:")
overall["auditor"] = {}
for dim, vals in global_scores["auditor"].items():
    if vals:
        avg = float(np.mean(vals))
        bar = "#" * int(avg * 3)
        print(f"    {dim:35s}: {avg:.2f}/10  {bar}")
        overall["auditor"][dim] = round(avg, 4)

print("\n  ARCHITECT:")
overall["architect"] = {}
for dim, vals in global_scores["architect"].items():
    if vals:
        avg = float(np.mean(vals))
        bar = "#" * int(avg * 3)
        print(f"    {dim:35s}: {avg:.2f}/10  {bar}")
        overall["architect"][dim] = round(avg, 4)

output["ALL_AUTHORS_AVERAGE"] = overall

with open("human_eval_averages.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("\nSaved: human_eval_averages.json")


HUMAN EVALUATION RESULTS — PER AUTHOR

User: Hashan

  AUDITOR:
    issue_identification               : 9.24/10  ###########################
    reasoning_quality                  : 8.45/10  #########################
    recommendation_actionability       : 8.55/10  #########################
    rubric_calibration                 : 7.22/10  #####################
    overall_audit_quality              : 8.08/10  ########################

  ARCHITECT:
    contract_alignment                 : 7.98/10  #######################
    fix_report_accuracy                : 8.47/10  #########################
    architecture_quality               : 7.51/10  ######################
    security_coverage                  : 7.24/10  #####################
    plan_completeness                  : 8.51/10  #########################

User: Yuvin

  AUDITOR:
    issue_identification               : 9.33/10  ############################
    reasoning_quality                  : 8.63/10  ###################